In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

import re
import string
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
pd.set_option('display.max_colwidth', None)
stop_words = set(stopwords.words('english'))

In [ ]:
#/kaggle/input/nlp-getting-started/sample_submission.csv
train=pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test=pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

In [ ]:
train.head()

In [ ]:
test.tail()

In [ ]:
train.shape, test.shape

In [ ]:
train.isna().sum()

In [ ]:
def clean_tweet(text):
    text=re.sub(r'http\S+|www\S+|https\S+', '', text)
    text=re.sub(r'@\w+', '', text)
    text=re.sub(r'#', '', text)
    text=re.sub(r'[^A-Za-z\s]', '', text)
    return text

In [ ]:
def dataPrepartion(df):
    df.drop(columns=['keyword', 'location'], inplace=True, errors='ignore')
    df['text'] = df['text'].str.lower()
    df['text'] = df['text'].apply(clean_tweet)
    df['tokens']=df['text'].apply(word_tokenize)
    df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])
    stemmer = PorterStemmer()
    df['tokens'] = df['tokens'].apply(lambda x: [stemmer.stem(word) for word in x])
    df['processed_text'] = df['tokens'].apply(lambda x: ' '.join(x))
    df.drop(columns=['text', 'tokens'], inplace=True, errors='ignore')

In [ ]:
dataPrepartion(train)
dataPrepartion(test)

In [ ]:
train

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
x_train = vectorizer.fit_transform(train['processed_text'])
x_test=vectorizer.fit_transform(test['processed_text'])
y_train=train['target']

In [ ]:
model=LogisticRegression()
model.fit(x_train, y_train)

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
output=pd.DataFrame({
    'id': test['id'],
    'target': y_pred
})
output.to_csv('submission.csv', index=False)

In [ ]:
output.head()